# 18 · Query Rewriting 查询改写

> 用户的问题往往口语化、指代不清，直接检索效果差。改写让“去检索的查询”与知识库口径更一致。

**本文件覆盖知识点**：Query Rewrite / Query Expansion / Query Normalization / Query Decomposition（概览）

```text
用户: Redis为什么这么快？
改写: → Redis single-thread architecture
     → Redis event loop
     → Redis memory management
     → Redis IO multiplexing
```

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 四种变换（先分清）

| 变换 | 在做什么 | 例子 |
|------|---------|------|
| **Rewrite（改写）** | 补全、转成更利于检索的表述 | 追问“那它贵吗”→ 补全为“客服机器人价格” |
| **Expansion（扩展）** | 一个查询 → 多个同义/相关查询 | “怎么收费” + “计费档位” + “价格套餐” |
| **Normalization（归一）** | 统一说法/去噪 | “咋弄”→“怎么做”；术语统一 |
| **Decomposition（拆解）** | 复杂问题拆成子问题 | “A与B如何保证可靠、区别是什么”→3问（第20课） |

> 底层机制都一样：用 LLM 生成（也可以用规则），把查询变成检索器更容易匹配的样子。

In [ ]:
# 知识点·真调说明：Query Expansion —— 把一条用户问题现场扩成多个同义/相关检索问法（Multi-Query 的源头）
import json as _json
_fall = '["星云客服机器人收费方式有哪些", "星云客服机器人价格套餐档位", "星云客服机器人按年还是按月计费"]'
out = _llm_live(
    prompt="""把下面这句用户问题，扩展成 3 条“意思相关、但措辞或侧重不同、更适合做关键词检索”的查询，只输出 JSON 字符串数组，不要输出任何解释。

问题：星云客服机器人怎么收费""",
    system='你是 RAG 检索查询扩展器。输出的 3 条必须从不同说法（收费=价格=计费=套餐）覆盖同一意图，且都不能与原文完全相同。',
    fallback=_fall,
    temperature=0.3,
)
s = out if out is not None else _fall
s = s.strip().strip('`')
if s.startswith('json'):
    s = s[4:].strip()
if out is None:
    print('（以上为固定样例；下面用样例走 json.loads 解析）')
try:
    qs = _json.loads(s)
    for i, qx in enumerate(qs, 1):
        print('  扩展查询%d：%s' % (i, qx))
    print('  检索时由 1 种说法 → %d 路并行走' % len(qs))
except Exception as e:
    print('未通过 json.loads：', e, '—— 说明结构化约束需再收紧')
print('→ 只按“收费”一种说法检索会漏掉写“价格/计费/套餐”的资料；扩写成多路查询各自召回再合并，Recall 更高——这就是 19 课 Multi-Query。')

In [ ]:
# 知识点·真调说明：Query Normalization —— 让模型把几条口语化、说法不一致的提问统一成规范检索词
import json as _json
_fall = ('[{"raw": "这玩意咋退钱？", "norm": "客服机器人如何申请退款"}, '
         '{"raw": "想申请把机器人退了，咋弄呀？", "norm": "客服机器人退货退款流程"}, '
         '{"raw": "你们退费咋操作啊，着急", "norm": "客服机器人退费操作步骤"}]')
out = _llm_live(
    prompt="""下面 3 条是用户对同一件事的不同问法，请把每条“归一化”成一条规范、书面、适合检索的查询：口语（咋弄/咋退钱/退费）统一成规范说法，去掉语气词和口水话。输出 JSON 数组，不解释。
1. 这玩意咋退钱？
2. 想申请把机器人退了，咋弄呀？
3. 你们退费咋操作啊，着急

JSON 格式：[{"raw": 原话原文, "norm": 归一化后的检索查询}]""",
    system='你是 RAG 查询归一化器：把口语、方言、语气词、同义说法统一成规范书面检索词（退钱/退费/退货 → 统一为“退款/退货退款”）。',
    fallback=_fall,
    temperature=0.2,
)
s = out if out is not None else _fall
s = s.strip().strip('`')
if s.startswith('json'):
    s = s[4:].strip()
if out is None:
    print('（以上为固定样例；下面用样例走 json.loads 解析）')
try:
    for it in _json.loads(s):
        print('  口语原话：%s' % it['raw'])
        print('  → 归一化：%s' % it['norm'])
except Exception as e:
    print('未通过 json.loads：', e, '—— 说明约束需收紧')
print('→ 归一让“五花八门的说法”收敛到同一规范检索词：无论用户怎么问，召回与后续精确匹配/过滤都用同一条查询。')

In [ ]:
# API 通过 .env 配置
from dotenv import load_dotenv; load_dotenv()
import os, json
from dashscope import Generation
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')

def llm(messages, model='qwen-plus'):
    r = Generation.call(model=model, messages=messages, api_key=API_KEY, result_format='message')
    return r.output.choices[0].message.content

# 改写：结合对话历史，把追问补全成独立完整问题（对话式 RAG 的关键）
history = [('用户','星云客服机器人支持哪些部署方式？'),
           ('助手','支持公有云 SaaS 与私有化两种部署方式。')]
prompt = (
    '多轮对话历史:\n' + '\n'.join(f'{r}: {m}' for r, m in history) + '\n\n'
    '把用户最新一句话改写成不依赖上文的独立检索问题，只输出问题本身。\n'
    '最新一句: 那它贵不贵？'
)
if API_KEY and '你的' not in API_KEY:
    print('改写结果:', llm([{'role':'user','content':prompt}]))
else:
    print('改写结果(示例): 星云客服机器人收费贵吗，价格大概是多少？')

## 2. 多轮对话里的查询改写

对话式 RAG 流程：
```text
用户追问
  → LLM(历史+追问) 改写为独立查询     ← 消解“它/那个”指代
  → 检索(独立查询)
  → 生成(带历史)
  → 写回历史
```

## 3. 工程细节

- 改写也要结构化输出（要求只返回 JSON/纯文本，别带解释）；
- 历史要截断（只留最近几轮/最近 N 个 token）；
- 可加“改写失败则原样返回”的兜底。

## 小结

- 改写解决指代和表述不匹配；扩展、拆解、归一解决的是别的问题；
- 底层都用 LLM 生成查询，注意结构化与兜底；
- 一个查询拆成多个并行召回，就是下一课的 Multi-Query。